# OTTO · Round 04 — build the query-safe demand index

**Use OTTO - Notebook.** This first bounded milestone scans existing raw TRAIN data once and preserves resumable units. No new experiment models, graph rebuilds, downloads or installations. Read the scope and leakage contract in RESEARCH_PLAN.md. Do not run other experiments concurrently.

In [1]:
from pathlib import Path
import importlib.util
import json
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'demand_features.py').is_file() and (p / 'launch.py').is_file()),
            Path.home() / 'otto_feature_round04')
if not (ROOT / 'demand_features.py').is_file():
    raise RuntimeError('Open this notebook from the extracted otto_feature_round04 folder')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location('otto_round04_launcher', ROOT / 'launch.py')
launch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launch)
sys.modules['launch'] = launch
state = {'halted': False, 'completed': []}
ML = Path.home() / 'otto-recommender-system/.venv/bin/python'
print('KERNEL_READY')
print('Notebook Python:', sys.executable)
print('ML Python:', ML)

def stage(name):
    if state['halted']:
        raise RuntimeError('Earlier stage stopped. Do not continue; collect the return ZIP.')
    try:
        value = launch.run_stage(name)
    except BaseException:
        state['halted'] = True
        raise
    state['completed'].append(name)
    return value


KERNEL_READY
Notebook Python: /opt/conda/bin/python
ML Python: /home/sagemaker-user/otto-recommender-system/.venv/bin/python


## Tests
Synthetic boundaries, full-catalogue priors, study-session exclusion, resumability, exact metrics and one tiny synthetic native-model reload.

In [2]:
stage('tests')

RUNNING tests; process cap 60s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round04/outputs/tests.log
test_action_mix_sums_one (test_demand.FeatureTests.test_action_mix_sums_one) ... ok
test_candidate_order_equivariance (test_demand.FeatureTests.test_candidate_order_equivariance) ... ok
test_count_monotonicity_rejected (test_demand.FeatureTests.test_count_monotonicity_rejected) ... ok
test_exact_75_unique_names_and_matrix (test_demand.FeatureTests.test_exact_75_unique_names_and_matrix) ... ok
test_fractional_counts_rejected (test_demand.FeatureTests.test_fractional_counts_rejected) ... ok
test_future_last_timestamp_rejected (test_demand.FeatureTests.test_future_last_timestamp_rejected) ... ok
test_input_nonmutation_and_replay (test_demand.FeatureTests.test_input_nonmutation_and_replay) ... ok
test_invalid_prior_and_query_time (test_demand.FeatureTests.test_invalid_prior_and_query_time) ... ok
test_last_seen_count_consistency (test_demand.FeatureTests.test_last_seen_

{'phase': 'tests', 'exit_code': 0}

## Build or resume the raw-training index
Only events before the largest fitting snapshot are admitted; every selected study session is excluded. The work limit is 300 seconds, outer process cap 320. A normal PAUSED_CHECKPOINTED preserves chunks. **One further manual index invocation** is allowed in this initial budget; a second pause or any failure requires review, not a loop.

In [3]:
stage('index')

RUNNING index; process cap 320s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round04/outputs/index.log
2026-09-12T00:57:31.633482+00:00 LAUNCHER_HEARTBEAT phase=index seconds=15.0
{"completed": 0, "elapsed_seconds": 15.0, "event": "heartbeat", "stage": "hash_permitted_raw_training", "total": 11307535945, "utc": "2026-09-12T00:57:31.707702+00:00"}
2026-09-12T00:57:46.645389+00:00 LAUNCHER_HEARTBEAT phase=index seconds=30.0
{"completed": 0, "elapsed_seconds": 30.0, "event": "heartbeat", "stage": "hash_permitted_raw_training", "total": 11307535945, "utc": "2026-09-12T00:57:46.707940+00:00"}
2026-09-12T00:58:01.657094+00:00 LAUNCHER_HEARTBEAT phase=index seconds=45.0
{"completed": 0, "elapsed_seconds": 45.0, "event": "heartbeat", "stage": "hash_permitted_raw_training", "total": 11307535945, "utc": "2026-09-12T00:58:01.708164+00:00"}
{"chunk": 0, "event": "raw_chunk_complete", "raw_bytes_done": 67114546, "raw_bytes_total": 11307535945, "seconds": 1.739, "source_sessions_

RuntimeError: PAUSED_CHECKPOINTED: valid completed units saved. Stop here and return the ZIP; no automatic retry.

## Completion gate
Open the comparison notebook only after this gate succeeds. The index is reusable; subsequent features must not reread raw sessions.

In [ ]:
if state['halted']:
    raise RuntimeError('Stop; return evidence before continuing')
s = json.loads((ROOT/'outputs/index_summary.json').read_text())
assert s['status'] == 'ROUND04_INDEX_READY'
print('INDEX_READY_FOR_FEATURE_COMPARISON')
print('Raw bytes processed:', s['raw_bytes_processed'])
print('Completed raw chunks:', s['raw_chunks'])
print('Study sessions excluded:', s['excluded_study_sessions'])
print('Event source:', s['availability'])
launch.collect()